In [400]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [401]:
data = pd.read_csv('train.csv')
data.head(10)

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [402]:
data.shape

(42000, 785)

In [403]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

print(data[:10])
data_dev = data[0:1000].T
Y_dev = data_dev[0]
print(Y_dev[:10])
X_dev = data_dev[1:n]
X_dev = X_dev / 255.

data_train = data[1000:m].T
print(data_train[:10])
Y_train = data_train[0]
print(Y_train[:10])
X_train = data_train[1:n] / 255.

[[5 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 [2 0 0 ... 0 0 0]
 ...
 [1 0 0 ... 0 0 0]
 [4 0 0 ... 0 0 0]
 [3 0 0 ... 0 0 0]]
[5 1 2 2 8 0 2 1 4 3]
[[2 0 8 ... 2 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
[2 0 8 6 8 2 7 8 1 7]


In [404]:
X_train.shape
Y_train

array([2, 0, 8, ..., 2, 0, 0], dtype=int64)

In [405]:
def init_params():
    W1 = np.random.rand(10,784) - 0.5
    b1 = np.random.rand(10,1) - 0.5
    W2 = np.random.rand(10,10) - 0.5
    b2 = np.random.rand(10,1) - 0.5

    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0,Z)

def derivative_relu(Z):
    return Z > 0 

def softmax(Z):
    Z -= np.max(Z,axis=0)
    return np.exp(Z)/np.sum(np.exp(Z), axis=0)

def forward_prop(W1, b1, W2, b2, X):
    Z1 = np.dot(W1,X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max()+1))
    one_hot_Y[np.arange(Y.size),Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y)

    dZ2 = A2 - one_hot_Y
    dW2 = 1/m * dZ2.dot(A1.T)
    db2 = 1/m * sum(dZ2)

    dZ1 = W2.T.dot( dZ2) * derivative_relu(Z1)
    dW1 = 1/m * dZ1.dot(X.T)
    db1 = 1/m * sum(dZ1)

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2

    return W1, b1, W2, b2


In [406]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    # print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2) 
            print(get_accuracy(predictions, Y) * 100)
    return W1, b1, W2, b2

In [413]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 500)

Iteration:  0
12.54390243902439
Iteration:  10
23.83170731707317
Iteration:  20
31.60487804878049
Iteration:  30
34.93170731707317
Iteration:  40
38.90243902439025
Iteration:  50
43.01951219512195
Iteration:  60
47.068292682926824
Iteration:  70
51.13414634146341
Iteration:  80
54.81219512195123
Iteration:  90
57.83658536585365
Iteration:  100
60.62439024390244
Iteration:  110
62.853658536585364
Iteration:  120
64.79512195121951
Iteration:  130
66.31951219512196
Iteration:  140
67.79756097560976
Iteration:  150
69.0780487804878
Iteration:  160
70.21219512195121
Iteration:  170
71.30243902439024
Iteration:  180
72.34390243902439
Iteration:  190
73.30731707317074
Iteration:  200
74.12439024390244
Iteration:  210
74.94390243902438
Iteration:  220
75.6780487804878
Iteration:  230
76.23658536585366
Iteration:  240
76.81219512195122
Iteration:  250
77.31463414634146
Iteration:  260
77.84390243902439
Iteration:  270
78.34390243902439
Iteration:  280
78.74634146341464
Iteration:  290
79.195121

In [432]:
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    # print(A2.shape)
    return predictions


def test_prediction(index, W1, b1, W2, b2):
    current_image = X_train[:, index, None]
    prediction = make_predictions(X_train[:, index, None], W1, b1, W2, b2)
    label = Y_train[index]
    print("Prediction: ", prediction)
    print("Label: ", label)
    
    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

make_predictions(X_train,W1,b1, W2, b2)

array([2, 0, 2, ..., 2, 0, 0], dtype=int64)

In [430]:
# print(X_train[:,:10].shape)

# test_prediction(1, W1, b1, W2, b2)
# import seaborn as sns
# print(Y_train[:10].shape, dev_prediction.shape)
# sns.heatmap((dev_prediction,Y_train[:10]))

In [431]:
dev_prediction = make_predictions(X_train,W1,b1, W2, b2)
get_accuracy(dev_prediction, Y_train)

0.8385121951219512